# Empirical Dataset Collection: Liquidity & OHLCV

This notebook is the foundation of the Empirical Resolution Rate (ERR) paper. 

**Methodology:**
1. **Load Universe:** We load the strictly verified xStocks discovered in our previous Jupiter scan.
2. **Liquidity Verification:** We query Birdeye to measure the exact US Dollar Liquidity resting in the AMM pools for each asset.
3. **OHLCV Extraction:** We extract the historical 15-minute price candles (Open, High, Low, Close, Volume) to empirically plot the weekend gaps.


In [1]:
import os
import json
import time
import httpx
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Load Birdeye API Key
load_dotenv("../.env")
BIRDEYE_API_KEY = os.getenv("BIRDEYE_API_KEY", "")
HEADERS = {"X-API-KEY": BIRDEYE_API_KEY, "x-chain": "solana"}

if not BIRDEYE_API_KEY:
    print("⚠️ WARNING: No Birdeye API Key found. The data requests will fail.")

# Load the verified asset universe we generated previously
try:
    with open("verified_xstocks.json", "r") as f:
        verified_assets = json.load(f)
    print(f"Loaded {len(verified_assets)} verified assets from previous scan.")
except FileNotFoundError:
    print("⚠️ ERROR: verified_xstocks.json not found. Run the xstock_scanner notebook first!")
    verified_assets = {}


Loaded 11 verified assets from previous scan.


## Phase 1: Total AMM Liquidity Snapshot
We must identify which assets actually have enough liquidity to execute empirical trades.


In [2]:
async def get_liquidity_snapshot():
    print("Fetching Liquidity from Birdeye...")
    liquidity_data = []
    
    async with httpx.AsyncClient() as client:
        for ticker, data in verified_assets.items():
            mint = data['mint']
            
            # Rate limit protection (Birdeye free tier is sensitive)
            await asyncio.sleep(1.5)
            
            try:
                res = await client.get(
                    f"https://public-api.birdeye.so/defi/token_overview?address={mint}",
                    headers=HEADERS
                )
                
                if res.status_code == 200:
                    info = res.json().get('data', {})
                    usd_liquidity = info.get('liquidity', 0)
                    price = info.get('price', 0)
                    volume_24h = info.get('v24h', 0)
                    
                    liquidity_data.append({
                        "Ticker": ticker,
                        "Provider": data['provider'],
                        "AMM": data['amm'],
                        "Price": price,
                        "Liquidity (USD)": usd_liquidity,
                        "24h Volume": volume_24h,
                        "Mint": mint
                    })
                else:
                    print(f"Failed to fetch {ticker}: {res.status_code}")
            except Exception as e:
                print(f"Error on {ticker}: {e}")
                
    # Create a nice Pandas DataFrame to view the data
    df = pd.DataFrame(liquidity_data)
    
    # Sort by Most Liquid
    if not df.empty:
        df = df.sort_values(by="Liquidity (USD)", ascending=False).reset_index(drop=True)
        
        # Formatting for display
        df['Price'] = df['Price'].apply(lambda x: f"${x:,.2f}")
        df['Liquidity (USD)'] = df['Liquidity (USD)'].apply(lambda x: f"${x:,.2f}")
        df['24h Volume'] = df['24h Volume'].apply(lambda x: f"${x:,.2f}")
    
    return df

import asyncio
df_liquidity = await get_liquidity_snapshot()
display(df_liquidity)


Fetching Liquidity from Birdeye...


,Ticker,Provider,AMM,Price,Liquidity (USD),24h Volume,Mint
0,AAPLX,Backed,Byreal,$304.30,"$440,333.86",$630.40,XsbEhLAtcf6HdfpFZ5xEMdqW8nfAvcsP5bdudRLJzJp
1,MSFTX,Backed,Illiquid/No Route,$496.05,"$247,989.02",$15.07,XspzcW1PRtgf6Wj92HCiZdjzKCyFekVD8P5Ueh3dRMX
2,AMATON,Ondo,Illiquid/No Route,$507.18,"$4,938.60",$0.00,7eRX747PSbVtGVx3qD5UFdkNM2BfTy86ikUiCMhondo
3,QCOMON,Ondo,Illiquid/No Route,$165.52,"$4,507.54",$0.00,hrmX7MV5hifoaBVjnrdpz698yABxrbBNAcWtWo9ondo
4,JNJON,Ondo,Illiquid/No Route,$259.73,"$3,294.61",$0.00,KUXt7LzHWSQXp5eyqMZRxWjAP6yM8BUh4LRHwiwondo
5,PGX,Backed,Illiquid/No Route,$146.38,"$2,901.41",$0.03,XsYdjDjNUygZ7yGKfQaB6TxLh2gC6RRjzLtLAGJrhzV
6,MCDON,Ondo,Illiquid/No Route,$273.38,"$2,100.27",$0.00,EUbJjmDt8JA222M91bVLZs211siZ2jzbFArH9N3ondo
7,AAPLON,Ondo,Illiquid/No Route,$306.48,$895.58,$2.31,123mYEnRLM2LLYsJW3K6oyYh8uP1fngj732iG638ondo
8,TMOX,Backed,Raydium CLMM,$423.65,$26.64,$0.00,Xs8drBWy3Sd5QY3aifG9kt9KFs2K3PGZmx7jWrsrk57
9,AMATX,Backed,FluxBeam,"$1,982.12",$2.02,$0.00,XsQZdaWUAGC4R3fgD2N1fupKvJfJq6YM51ccnsLUWFA


## Phase 2: Historical OHLCV Extraction (Weekend Gaps)
Now we select the most liquid asset from our list and download the 15-minute price candles for the last 7 days to analyze the weekend gap.


In [3]:
async def get_ohlcv_data(target_mint):
    # We want data from 7 days ago to right now
    end_time = int(datetime.now().timestamp())
    start_time = int((datetime.now() - timedelta(days=7)).timestamp())
    
    print(f"Fetching 15m Candles for the last 7 days...")
    
    async with httpx.AsyncClient() as client:
        res = await client.get(
            f"https://public-api.birdeye.so/defi/ohlcv",
            params={
                "address": target_mint,
                "type": "15m", # 15 minute candles
                "time_from": start_time,
                "time_to": end_time
            },
            headers=HEADERS
        )
        
        if res.status_code == 200:
            items = res.json().get('data', {}).get('items', [])
            
            # Convert to Pandas DataFrame
            df_candles = pd.DataFrame(items)
            
            # Format the unix timestamps into human readable dates
            df_candles['datetime'] = pd.to_datetime(df_candles['unixTime'], unit='s')
            
            # Reorder columns for standard financial analysis
            df_candles = df_candles[['datetime', 'o', 'h', 'l', 'c', 'v']]
            df_candles.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
            
            return df_candles
        else:
            print("Failed to fetch OHLCV:", res.text)
            return pd.DataFrame()

# Automatically select the MOST liquid asset from our previous step
if not df_liquidity.empty:
    most_liquid_mint = df_liquidity.iloc[0]['Mint']
    most_liquid_ticker = df_liquidity.iloc[0]['Ticker']
    
    print(f"Targeting: {most_liquid_ticker}")
    df_history = await get_ohlcv_data(most_liquid_mint)
    
    # Show the last 5 candles
    display(df_history.tail(5))
    
    # Save to CSV for the research paper
    csv_filename = f"{most_liquid_ticker}_ohlcv.csv"
    df_history.to_csv(csv_filename, index=False)
    print(f"\n✅ Data saved to {csv_filename} for Empirical Analysis!")
else:
    print("No liquidity data available.")


Targeting: AAPLX
Fetching 15m Candles for the last 7 days...


,Date,Open,High,Low,Close,Volume
667,2026-08-16 13:30:00,305.475191,306.721097,304.791384,305.463075,0.651383
668,2026-08-16 13:45:00,305.463075,305.463075,305.363492,305.461276,0.024694
669,2026-08-16 14:00:00,305.461276,306.759302,305.417761,306.759302,1.094153
670,2026-08-16 14:15:00,306.759302,306.770103,305.540607,305.540888,0.055900
671,2026-08-16 14:30:00,305.540888,305.540888,305.293979,305.460346,2.883983



✅ Data saved to AAPLX_ohlcv.csv for Empirical Analysis!
